# 02 · Baseline, fine-tuning QLoRA, avaliação e exportação

Corre com **Runtime → Change runtime type → T4 GPU**.
Precisa de `train.jsonl`, `val.jsonl` e `test.jsonl` em `s3://$DATA_BUCKET/v1/`.
Os checkpoints vão para o Drive: se o Colab desligar, corre as células 1–2 e depois a 4, e o treino retoma.

## 1. Preparar o ambiente

In [ ]:
# Instalar dependências (awscli: o Colab não traz a CLI da AWS)
!pip install -q -U transformers trl peft bitsandbytes accelerate datasets boto3 pydantic awscli

import os
from google.colab import drive, userdata

# Secrets do Colab (ícone da chave, à esquerda): AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY, DATA_BUCKET
for k in ("AWS_ACCESS_KEY_ID", "AWS_SECRET_ACCESS_KEY", "DATA_BUCKET"):
    os.environ[k] = userdata.get(k).strip()
os.environ["AWS_DEFAULT_REGION"] = "eu-west-1"

drive.mount("/content/drive")  # tudo o que demora vai para o Drive: sobrevive a desconexões
DRIVE = "/content/drive/MyDrive/wiki-bio-distill"
os.makedirs(DRIVE, exist_ok=True)

# Repositório: público, ou privado com um secret GITHUB_TOKEN
try:
    token = userdata.get("GITHUB_TOKEN")
    url = f"https://{token}@github.com/pinthoz/wiki-bio-distill.git"
except Exception:
    url = "https://github.com/pinthoz/wiki-bio-distill.git"
if not os.path.exists("/content/repo"):
    !git clone -q {url} /content/repo
%cd /content/repo
!git pull -q
!mkdir -p data preds
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!for f in train val test; do aws s3 cp s3://$DATA_BUCKET/v1/$f.jsonl data/; done
!wc -l data/*.jsonl

BASE = "Qwen/Qwen2.5-1.5B-Instruct"
RUN_DIR = f"{DRIVE}/run-v1"

## 2. Baseline: o modelo base sem treino, com o prompt detalhado

In [ ]:
import json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from common import SYSTEM_TEACHER
from gen import generate_preds

tokenizer = AutoTokenizer.from_pretrained(BASE)
base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
test = [json.loads(l) for l in open("data/test.jsonl")]

generate_preds(base, tokenizer, test, SYSTEM_TEACHER, "preds/base.jsonl")
!python evaluate.py preds/base.jsonl
!head -n 3 preds/base.jsonl

## 3. Libertar a GPU

In [ ]:
import gc
del base; gc.collect(); torch.cuda.empty_cache()

## 4. Fine-tuning QLoRA (~30–60 min numa T4)

In [ ]:
import glob
from datasets import load_dataset
from peft import LoraConfig
from transformers import BitsAndBytesConfig
from trl import SFTConfig, SFTTrainer

model = AutoModelForCausalLM.from_pretrained(
    BASE,
    quantization_config=BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16),   # a T4 não suporta bfloat16
    device_map="auto",
)
data = load_dataset("json", data_files={"train": "data/train.jsonl", "validation": "data/val.jsonl"})

peft_config = LoraConfig(r=16, lora_alpha=32, lora_dropout=0.05,
                         target_modules="all-linear", task_type="CAUSAL_LM")
args = SFTConfig(
    output_dir=RUN_DIR, num_train_epochs=2,
    per_device_train_batch_size=8, gradient_accumulation_steps=2,
    learning_rate=2e-4, lr_scheduler_type="cosine", warmup_steps=10,
    max_length=1024, fp16=True, gradient_checkpointing=True,
    logging_steps=10, eval_strategy="steps", eval_steps=50, save_steps=50, save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", report_to="none",
)
trainer = SFTTrainer(model=model, args=args, train_dataset=data["train"],
                     eval_dataset=data["validation"], processing_class=tokenizer,
                     peft_config=peft_config)
trainer.model.print_trainable_parameters()

resume = bool(glob.glob(f"{RUN_DIR}/checkpoint-*"))   # retoma depois de uma desconexão
trainer.train(resume_from_checkpoint=resume)
trainer.save_model(f"{RUN_DIR}/adapter")
tokenizer.save_pretrained(f"{RUN_DIR}/adapter")

In [ ]:
!aws s3 cp --recursive {RUN_DIR}/adapter s3://$DATA_BUCKET/v1/models/qwen1.5b-lora-r16/adapter/

## 5. Avaliar o aluno (merge do adaptador em fp16)

In [ ]:
from peft import PeftModel
from common import SYSTEM_STUDENT

del model, trainer; gc.collect(); torch.cuda.empty_cache()

base = AutoModelForCausalLM.from_pretrained(BASE, dtype=torch.float16, device_map="auto")
student = PeftModel.from_pretrained(base, f"{RUN_DIR}/adapter").merge_and_unload()

generate_preds(student, tokenizer, test, SYSTEM_STUDENT, "preds/student.jsonl")
!python evaluate.py preds/base.jsonl preds/student.jsonl

## 6. Análise de erros

In [ ]:
from common import FIELDS, as_set, parse_json

preds = {p["id"]: parse_json(p["raw"]) for p in map(json.loads, open("preds/student.jsonl"))}
shown = 0
for t in test:
    p = preds[t["id"]] or {}
    wrong = [f for f in FIELDS if as_set(p.get(f)) != as_set(t["gold"].get(f))]
    if wrong and shown < 10:
        shown += 1
        print(t["text"][:200], "…")
        for f in wrong:
            print(f"  {f}: ouro={t['gold'].get(f)!r}  aluno={p.get(f)!r}  professor={t['teacher'].get(f)!r}")
        print()

## 7. Exportar para GGUF e quantizar

Só se instala o pacote `gguf` do llama.cpp. **Não** corras o `requirements.txt` do llama.cpp neste ambiente: instala o PyTorch de CPU e uma versão antiga do transformers.

In [ ]:
MERGED = "/content/merged"
student.save_pretrained(MERGED)
tokenizer.save_pretrained(MERGED)
del student, base; gc.collect(); torch.cuda.empty_cache()

!git clone -q --depth 1 https://github.com/ggml-org/llama.cpp /content/llama.cpp
!pip install -q /content/llama.cpp/gguf-py sentencepiece

!python /content/llama.cpp/convert_hf_to_gguf.py {MERGED} --outtype f16 --outfile /content/student-f16.gguf
!cmake -S /content/llama.cpp -B /content/llama.cpp/build -DGGML_CUDA=OFF > /dev/null
!cmake --build /content/llama.cpp/build --target llama-quantize -j > /dev/null
!/content/llama.cpp/build/bin/llama-quantize /content/student-f16.gguf /content/student-q4_k_m.gguf Q4_K_M
!ls -lh /content/*.gguf

## 8. Medir a perda da quantização (CPU, com gramática JSON como na Lambda)

In [ ]:
!pip install -q llama-cpp-python==0.3.35   # compila em CPU: alguns minutos

import time
from llama_cpp import Llama
from common import SCHEMA, messages

llm = Llama(model_path="/content/student-q4_k_m.gguf", n_ctx=2048, verbose=False)
with open("preds/student-q4.jsonl", "w") as f:
    for t in test:
        t0 = time.perf_counter()
        out = llm.create_chat_completion(
            messages=messages(t["text"], SYSTEM_STUDENT), temperature=0, max_tokens=300,
            response_format={"type": "json_object", "schema": SCHEMA})
        f.write(json.dumps({"id": t["id"], "raw": out["choices"][0]["message"]["content"],
                            "ms": round((time.perf_counter() - t0) * 1000)}, ensure_ascii=False) + "\n")

!python evaluate.py preds/base.jsonl preds/student.jsonl preds/student-q4.jsonl

## 9. Guardar tudo no S3

In [ ]:
!cd /content && sha256sum student-q4_k_m.gguf > student-q4_k_m.gguf.sha256
!aws s3 cp /content/student-q4_k_m.gguf        s3://$DATA_BUCKET/v1/models/qwen1.5b-lora-r16/
!aws s3 cp /content/student-q4_k_m.gguf.sha256 s3://$DATA_BUCKET/v1/models/qwen1.5b-lora-r16/
!aws s3 cp preds/ s3://$DATA_BUCKET/v1/preds/ --recursive
!aws s3 cp results.json s3://$DATA_BUCKET/v1/results.json